# Pátio Monitor · Avaliação de IA generativa

As saídas abaixo foram executadas sobre a avaliação com Qwen2.5-0.5B-Instruct local. O JSON de entrada conserva as 51 respostas, incluindo recusas, latência e fontes. Abra este notebook a partir de `gs/genai` ou da raiz do repositório.

Para refazer a inferência, instale as dependências e baixe o peso de acordo com o README, configure `PATIO_LLM_MODEL_PATH` e rode `python gs/genai/evaluate.py` a partir da raiz. Estas células analisam os resultados gerados.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
data_file = root / 'evaluation.json'
if not data_file.exists():
    data_file = root / 'gs' / 'genai' / 'evaluation.json'
report = json.loads(data_file.read_text(encoding='utf-8'))
print('Modelo:', report['model_name'])
print('SHA-256:', report['model_sha256'])
print('Threads:', report['threads'], '| camadas GPU:', report['n_gpu_layers'])
print('Temperatura:', report['temperature'], '| seed:', report['seed'])
print('Casos por método:', report['retrieval']['n'])

Modelo: qwen2.5-0.5b-instruct
SHA-256: 74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db
Threads: 1 | camadas GPU: 0
Temperatura: 0 | seed: 42
Casos por método: 17


In [2]:
retriever = report['retrieval']
calibration = report['calibration']
cutoff = report['threshold']
print(f"Recall@1 = {retriever['recall_at_1']:.1%}")
print(f"Recall@3 = {retriever['recall_at_3']:.1%}")
print(f"Perguntas gabaritadas com fonte: {retriever['answerable']}")
print('Limiar:', cutoff)
print('Calibração:', next(row for row in calibration['threshold_sweep'] if row['threshold'] == cutoff))

Recall@1 = 100.0%
Recall@3 = 100.0%
Perguntas gabaritadas com fonte: 11
Limiar: 0.46
Calibração: {'threshold': 0.46, 'answerable_pass': 7, 'unknown_blocked': 4}


In [3]:
for mode, result in report['scoreboard'].items():
    print(mode)
    for metric, value in result.items():
        print(f'  {metric}: {value}')

baseline
  correct_total: 4
  correct_answerable: 0
  correct_refusals: 4
  correct_with_relevant_citation: 0
  raw_model_json_valid: 17
  raw_model_calls: 17
  api_json_valid: 17
  median_seconds: 1.605
rag
  correct_total: 8
  correct_answerable: 2
  correct_refusals: 6
  correct_with_relevant_citation: 2
  raw_model_json_valid: 17
  raw_model_calls: 17
  api_json_valid: 17
  median_seconds: 4.868
rag_agent
  correct_total: 11
  correct_answerable: 5
  correct_refusals: 6
  correct_with_relevant_citation: 5
  raw_model_json_valid: 12
  raw_model_calls: 12
  api_json_valid: 17
  median_seconds: 4.007


In [4]:
sample = next(item for item in report['results'] if item['id'] == 'g17' and item['mode'] == 'rag_agent')
print('Pergunta:', sample['question'])
print('Fontes recuperadas:', sample['retrieved'])
print(json.dumps(sample['answer'], ensure_ascii=False, indent=2))

Pergunta: Que evidências devo verificar e qual é o próximo passo para revisar esta ocorrência?
Fontes recuperadas: [['p01', 1.0], ['s01', 0.4360677959668634], ['p08', 0.29224990825107466]]
{
  "answer": "Para revisar, conferir o registro OUT, o veículo e o horário, e manter a pendência se a evidência não for suficiente.",
  "citations": [
    {
      "id": "p01",
      "title": "Pátio Monitor · Evidência de passagem",
      "url": "https://github.com/MachadONLY/patio-monitor-ia-generativa/blob/main/README.md#fechamento-de-turno"
    }
  ],
  "refused": false,
  "actions": [
    "selecionar_busca",
    "recuperar_trechos",
    "restringir_à_fonte_recuperada",
    "gerar_com_llm_local",
    "verificar_citações",
    "validar_dados_numéricos"
  ],
  "model": "Qwen2.5-0.5B-Instruct Q4_K_M"
}


In [5]:
failures = [item for item in report['results'] if not item['correct']]
for case_id in ['g01', 'g03', 'g05', 'g09', 'g11']:
    item = next(row for row in failures if row['id'] == case_id)
    print(item['mode'], item['id'], item['question'])
    print('  Gabarito:', item['expected'])
    print('  Saída:', item['answer']['answer'])

baseline g01 Sem confirmação na portaria, GPS sozinho comprova passagem física pelo portão?
  Gabarito: Não. GPS é auxiliar e deve-se verificar o registro OUT.
  Saída: A informação solicitada não consta nas fontes disponíveis.
baseline g03 Sem ID da saída, que janela de minutos usa o Pátio Monitor para associar portaria OUT ao mesmo veículo?
  Gabarito: De 15 minutos antes até 60 minutos depois do horário previsto, com unicidade.
  Saída: A informação solicitada não consta nas fontes disponíveis.
baseline g05 Qual modo usa cenário sintético e qual exige observações reais do Olho Vivo?
  Gabarito: DEMO usa cenário sintético e LIVE exige observações reais.
  Saída: A informação solicitada não consta nas fontes disponíveis.
baseline g09 No Olho Vivo, o que os campos py e px representam?
  Gabarito: py é latitude e px é longitude.
  Saída: py é o valor de um campo e px é o valor de outro
baseline g11 Qual é a placa do ônibus 50730?
  Gabarito: Não consta.
  Saída: 50730
